# Metrics Analysis Notebook

This notebook allows you to analyze the performance metrics generated by the DatawiseAgent system. It reads a JSONL file (typically `experience_task.jsonl`) and calculates average latencies and token counts across all completed tasks.

In [ ]:
# Cell 1: Imports and Setup
# Ensure plots are displayed inline in Jupyter
%matplotlib inline

import json
from pathlib import Path
import os # Needed for os.path.exists and os.path.getsize, etc.
import matplotlib.pyplot as plt # Import for plotting
import numpy as np # For potential numerical operations like rolling averages

## Configuration

Specify the path to your `experience_task.jsonl` file below. Make sure the file exists.

In [ ]:
# Cell 2: User Input - Specify the path to your metrics file
# Example: file_path = "./evaluation/experimental_results/MatplotBench/cac_llm/experience_task.jsonl"
# file_path = "<PROJECT_ROOT>/evaluation/experimental_results/MatplotBench/moa_llm/experience_task.jsonl" # <<< --- CHANGE THIS PATH
file_path = "<PROJECT_ROOT>/evaluation/experimental_results/MatplotBench/cac_llm/experience_task.jsonl"

## `analyze_metrics` Function Definition

This function reads the specified JSONL file, filters for task summary records (`step_id == 'end'`), accumulates various latency and token count metrics, and then prints their averages. It also collects these metrics per task for plotting.

In [ ]:
# Cell 3: Function Definition
def analyze_metrics(file_path):
    # Lists to store metrics for each task
    metrics_data = {
        'task_indices': [],
        'avg_planner_call_count': [],
        'step_num': [],
        'call_count': [],
        'agent_failure_rate': [],
        'network_failure_rate': [],
        'agent_token_count': [],
        'router_token_count': [],
        'total_token_count': [],
        'agent_latency': [],
        'router_latency': [],
        'network_latency': [],
        'total_latency': []
    }

    # Initialize totals for average calculation (still useful for printing averages)
    total_agent_latency = 0
    total_router_latency = 0
    total_network_latency = 0
    total_total_latency = 0
    total_agent_token_count = 0
    total_router_token_count = 0
    total_total_token_count = 0

    task_idx = 0 # Use task_idx instead of task_count to match previous logic and for plotting index

    # Check if file exists and is not empty
    if not Path(file_path).exists():
        print(f"Error: File not found at {file_path}")
        return None # Return None if file not found
    if Path(file_path).stat().st_size == 0:
        print(f"Error: File {file_path} is empty.")
        return None # Return None if file is empty

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                task_data = json.loads(line)

                if task_data.get('step_id') == 'end':
                    # Accumulate for averages
                    total_agent_latency += task_data.get('agent_latency', 0)
                    total_router_latency += task_data.get('router_latency', 0)
                    total_network_latency += task_data.get('network_latency', 0)
                    total_total_latency += task_data.get('total_latency', 0)

                    total_agent_token_count += task_data.get('agent_token_count', 0)
                    total_router_token_count += task_data.get('router_token_count', 0)
                    total_total_token_count += task_data.get('total_token_count', 0)

                    # Store for plotting
                    metrics_data['task_indices'].append(task_idx)
                    metrics_data['avg_planner_call_count'].append(task_data.get('avg_planner_call_count', 0))
                    metrics_data['step_num'].append(task_data.get('step_num', 0))
                    metrics_data['call_count'].append(task_data.get('call_count', 0))
                    metrics_data['agent_failure_rate'].append(task_data.get('agent_failure_rate', 0))
                    metrics_data['network_failure_rate'].append(task_data.get('network_failure_rate', 0))
                    metrics_data['agent_token_count'].append(task_data.get('agent_token_count', 0))
                    metrics_data['router_token_count'].append(task_data.get('router_token_count', 0))
                    metrics_data['total_token_count'].append(task_data.get('total_token_count', 0))
                    metrics_data['agent_latency'].append(task_data.get('agent_latency', 0))
                    metrics_data['router_latency'].append(task_data.get('router_latency', 0))
                    metrics_data['network_latency'].append(task_data.get('network_latency', 0))
                    metrics_data['total_latency'].append(task_data.get('total_latency', 0))
                    task_idx += 1
            except json.JSONDecodeError as e:
                print(f"Skipping malformed JSON line: {line.strip()} - Error: {e}")
            except KeyError as e:
                print(f"Skipping line due to missing key: {e} in line: {line.strip()}")

    if task_idx == 0:
        print("No tasks found or processed in the file.")
        return None # Return None if file not found

    # Print Averages
    print(f"Analysis of {file_path}:\n")
    print(f"Total tasks: {task_idx}")
    print(f"\nAverage Latencies:")
    print(f"  Agent Latency: {total_agent_latency / task_idx:.2f} ms")
    print(f"  Router Latency: {total_router_latency / task_idx:.2f} ms")
    print(f"  Network Latency: {total_network_latency / task_idx:.2f} ms")
    print(f"  Total Latency: {total_total_latency / task_idx:.2f} ms")
    print(f"\nAverage Token Counts:")
    print(f"  Agent Token Count: {total_agent_token_count / task_idx:.2f}")
    print(f"  Router Token Count: {total_router_token_count / task_idx:.2f}")
    print(f"  Total Token Count: {total_total_token_count / task_idx:.2f}")
    print(f"\nAverage Failure Rates:")
    print(f"  Agent Failure Rate: {sum(metrics_data['agent_failure_rate']) / task_idx:.4f}")
    print(f"  Network Failure Rate: {sum(metrics_data['network_failure_rate']) / task_idx:.4f}")
    print(f"\nAverage Step/Call Counts:")
    print(f"  Avg Planner Call Count: {sum(metrics_data['avg_planner_call_count']) / task_idx:.2f}")
    print(f"  Avg Step Num: {sum(metrics_data['step_num']) / task_idx:.2f}")
    print(f"  Avg Total Call Count: {sum(metrics_data['call_count']) / task_idx:.2f}")

    return metrics_data # Return the collected data

## Execute Analysis and Plot Metrics

Run this cell to perform the analysis and generate plots for the specified metrics.

You can customize:
- `rolling_window`: For smoothing the curves.
- `figsize`: Size of the plots.

In [ ]:
# Cell 4: Execute Analysis and Plotting

# Optional: Set a rolling window for smoothing curves (e.g., 5 for average over 5 tasks)
rolling_window = 5 # Set to 1 for no smoothing
figsize = (12, 6)

print("--- Starting metrics analysis ---")

metrics_results = analyze_metrics(Path(file_path))

if metrics_results is None:
    print("No data was returned from analyze_metrics. Please check your file_path and content.")
elif not metrics_results['task_indices']:
    print("Metrics results dictionary is empty. No tasks found to plot.")
else:
    print(f"Successfully loaded data for {len(metrics_results['task_indices'])} tasks.")
    task_indices = np.array(metrics_results['task_indices'])
    num_tasks = len(task_indices)

    # Adjust rolling_window if it's too large
    current_rolling_window = min(rolling_window, num_tasks) if num_tasks > 0 else 1
    if current_rolling_window < 1: # Ensure minimum window size
        current_rolling_window = 1
    if rolling_window > num_tasks and num_tasks > 0:
        print(f"Warning: rolling_window ({rolling_window}) is larger than the number of tasks ({num_tasks}). Adjusting to {current_rolling_window}.")
    elif num_tasks == 0:
        print("No tasks available for plotting. Skipping plots.")
        # Exit here if no tasks to avoid errors in plotting
    else:
        # Helper function for robust convolution
        def robust_convolve(data_list, window):
            if len(data_list) == 0: return np.array([])
            if window <= 1 or window > len(data_list):
                # If window is 1 or too large, no smoothing or plot raw
                return np.array(data_list)
            return np.convolve(data_list, np.ones(window)/window, mode='valid')

        # --- Plot Call Counts and Step Numbers ---
        plt.figure(figsize=figsize)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['avg_planner_call_count'], current_rolling_window), label='Avg Planner Call Count', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['step_num'], current_rolling_window), label='Step Num', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['call_count'], current_rolling_window), label='Total Call Count', alpha=0.7)
        plt.title('Agent Call Counts & Step Numbers Over Tasks')
        plt.xlabel('Task Index')
        plt.ylabel('Count')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # --- Plot Failure Rates ---
        plt.figure(figsize=figsize)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['agent_failure_rate'], current_rolling_window), label='Agent Failure Rate', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['network_failure_rate'], current_rolling_window), label='Network Failure Rate', alpha=0.7)
        plt.title('Failure Rates Over Tasks')
        plt.xlabel('Task Index')
        plt.ylabel('Rate')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # --- Plot Token Counts ---
        plt.figure(figsize=figsize)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['agent_token_count'], current_rolling_window), label='Agent Token Count', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['router_token_count'], current_rolling_window), label='Router Token Count', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['total_token_count'], current_rolling_window), label='Total Token Count', alpha=0.7)
        plt.title('Token Counts Over Tasks')
        plt.xlabel('Task Index')
        plt.ylabel('Tokens')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # --- Plot Latencies ---
        plt.figure(figsize=figsize)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['agent_latency'], current_rolling_window), label='Agent Latency (ms)', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['router_latency'], current_rolling_window), label='Router Latency (ms)', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['network_latency'], current_rolling_window), label='Network Latency (ms)', alpha=0.7)
        plt.plot(task_indices[current_rolling_window-1:] if current_rolling_window > 1 else task_indices, robust_convolve(metrics_results['total_latency'], current_rolling_window), label='Total Latency (ms)', alpha=0.7)
        plt.title('Latencies Over Tasks')
        plt.xlabel('Task Index')
        plt.ylabel('Latency (ms)')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

## Metric Comparison

This section allows you to compare a specific metric between two different experiment runs (JSONL files). The function `compare_metrics` will load data from two specified files, extract the chosen metric, and plot them on the same graph for visual comparison.

In [ ]:
# Cell 5: Function Definition for Metric Comparison
def compare_metrics(file_path1, file_path2, metric_name, label1="Experiment 1", label2="Experiment 2", rolling_window=1, figsize=(12, 6)):
    print(f"--- Starting comparison for metric: {metric_name} ---")

    metrics_results1 = analyze_metrics(Path(file_path1))
    metrics_results2 = analyze_metrics(Path(file_path2))

    if metrics_results1 is None or metrics_results2 is None:
        print("One or both files could not be analyzed. Skipping comparison.")
        return

    if metric_name not in metrics_results1 or metric_name not in metrics_results2:
        print(f"Error: Metric '{metric_name}' not found in one or both datasets.")
        print(f"Available metrics in file 1: {list(metrics_results1.keys())}")
        print(f"Available metrics in file 2: {list(metrics_results2.keys())}")
        return

    task_indices1 = np.array(metrics_results1['task_indices'])
    task_indices2 = np.array(metrics_results2['task_indices'])
    num_tasks1 = len(task_indices1)
    num_tasks2 = len(task_indices2)

    if num_tasks1 == 0 or num_tasks2 == 0:
        print("No tasks available for plotting in one or both files. Skipping plots.")
        return

    # Adjust rolling_window if it's too large for either dataset
    current_rolling_window = min(rolling_window, num_tasks1, num_tasks2) if min(num_tasks1, num_tasks2) > 0 else 1
    if current_rolling_window < 1:
        current_rolling_window = 1
    if rolling_window > min(num_tasks1, num_tasks2) and min(num_tasks1, num_tasks2) > 0:
        print(f"Warning: rolling_window ({rolling_window}) is larger than the number of tasks in one or both datasets. Adjusting to {current_rolling_window}.")

    # Helper function for robust convolution (copied from previous cell to ensure self-contained)
    def robust_convolve(data_list, window):
        if len(data_list) == 0: return np.array([])
        if window <= 1 or window > len(data_list):
            return np.array(data_list)
        return np.convolve(data_list, np.ones(window)/window, mode='valid')

    plt.figure(figsize=figsize)
    
    # Plot data from file 1
    plt.plot(task_indices1[current_rolling_window-1:] if current_rolling_window > 1 else task_indices1, 
             robust_convolve(metrics_results1[metric_name], current_rolling_window), 
             label=f'{label1} - {metric_name}', alpha=0.7)

    # Plot data from file 2
    plt.plot(task_indices2[current_rolling_window-1:] if current_rolling_window > 1 else task_indices2, 
             robust_convolve(metrics_results2[metric_name], current_rolling_window), 
             label=f'{label2} - {metric_name}', alpha=0.7)

    plt.title(f'{metric_name} Comparison Over Tasks')
    plt.xlabel('Task Index')
    plt.ylabel(metric_name.replace('_', ' ').title())
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 6: Execute Metric Comparison
# Define the paths to the two experience_task.jsonl files you want to compare
file_path_exp1 = "<PROJECT_ROOT>/evaluation/experimental_results/MatplotBench/cac_llm/experience_task.jsonl"
file_path_exp2 = "<PROJECT_ROOT>/evaluation/experimental_results/MatplotBench/moa_llm/experience_task.jsonl"

# Define the metric you want to compare (e.g., 'total_latency', 'total_token_count', 'agent_failure_rate')
metric_to_compare = 'total_latency' # <<< --- CHANGE THIS METRIC

# Optional: Labels for the legend
label_exp1 = "CAC"
label_exp2 = "MOA"

# Optional: Rolling window for smoothing (set to 1 for no smoothing)
comparison_rolling_window = 5

# Execute the comparison
compare_metrics(file_path_exp1, file_path_exp2, metric_to_compare, 
                label1=label_exp1, label2=label_exp2, 
                rolling_window=comparison_rolling_window)

metric_to_compare = 'total_token_count'
compare_metrics(file_path_exp1, file_path_exp2, metric_to_compare, 
                label1=label_exp1, label2=label_exp2, 
                rolling_window=comparison_rolling_window)

metric_to_compare = 'agent_failure_rate'
compare_metrics(file_path_exp1, file_path_exp2, metric_to_compare, 
                label1=label_exp1, label2=label_exp2, 
                rolling_window=comparison_rolling_window)
